# 01 — Extracción inicial desde TMDB

Este notebook comprueba que la API responde y busca la película principal del caso. No incluye claves dentro del código: las lee desde el archivo local `.env`.

## Objetivo de este primer paso

1. Conectarnos a TMDB con una clave local.
2. Buscar *A Minecraft Movie* (2025).
3. Revisar el resultado antes de elegir el identificador de la película.

Todavía no se hacen comparaciones ni conclusiones de negocio.

In [1]:
from datetime import datetime, timezone
import json
from pathlib import Path
import os

import pandas as pd
import requests
from dotenv import load_dotenv


In [2]:
# Encontramos la raíz del proyecto tanto si el notebook se ejecuta desde
# la carpeta notebooks/ como desde la raíz del repositorio.
def encontrar_raiz_proyecto(inicio: Path) -> Path:
    for carpeta in (inicio, *inicio.parents):
        if (carpeta / '.env.example').exists():
            return carpeta
    raise FileNotFoundError('No se encontró .env.example en las carpetas superiores.')

RAIZ_PROYECTO = encontrar_raiz_proyecto(Path.cwd().resolve())
# override=True recarga una clave que se haya agregado o corregido durante esta sesión.
load_dotenv(RAIZ_PROYECTO / '.env', override=True)

TMDB_API_KEY = os.getenv('TMDB_API_KEY')
OMDB_API_KEY = os.getenv('OMDB_API_KEY')
if not TMDB_API_KEY or not OMDB_API_KEY:
    raise ValueError('Falta una clave en .env. Completá TMDB_API_KEY y OMDB_API_KEY localmente.')

print('Claves de TMDB y OMDb detectadas localmente.')

Claves de TMDB y OMDb detectadas localmente.


In [3]:
BASE_URL_TMDB = 'https://api.themoviedb.org/3'

def consultar_tmdb(endpoint: str, **params) -> dict:
    """Hace una consulta GET a TMDB y detiene el notebook si ocurre un error."""
    respuesta = requests.get(
        f'{BASE_URL_TMDB}{endpoint}',
        params={'api_key': TMDB_API_KEY, **params},
        timeout=30,
    )
    respuesta.raise_for_status()
    return respuesta.json()


In [4]:
busqueda_minecraft = consultar_tmdb(
    '/search/movie',
    query='A Minecraft Movie',
    year=2025,
    language='es-AR',
)

resultados = pd.DataFrame(busqueda_minecraft['results'])
columnas_a_revisar = [
    columna
    for columna in ['id', 'title', 'original_title', 'release_date', 'popularity', 'vote_average', 'vote_count']
    if columna in resultados.columns
]
resultados[columnas_a_revisar]

,id,title,original_title,release_date,popularity,vote_average,vote_count
0,950387,Una película de Minecraft,A Minecraft Movie,2025-03-31,18.9597,6.191,3441


## Control de la película principal

La búsqueda confirmó que la fila correcta corresponde a *A Minecraft Movie* (2025), con identificador TMDB `950387`. En las siguientes celdas consultamos sus detalles y la fuente complementaria OMDb.

In [5]:
PELICULA_ID_TMDB = 950387
detalles_tmdb = consultar_tmdb(
    f'/movie/{PELICULA_ID_TMDB}',
    language='es-AR',
)

campos_tmdb = ['id', 'title', 'original_title', 'release_date', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'imdb_id']
pd.DataFrame([{campo: detalles_tmdb.get(campo) for campo in campos_tmdb}])

,id,title,original_title,release_date,budget,revenue,popularity,vote_average,vote_count,imdb_id
0,950387,Una película de Minecraft,A Minecraft Movie,2025-03-31,150000000,960387780,18.9597,6.192,3440,tt3566834


In [6]:
URL_OMDB = 'https://www.omdbapi.com/'

respuesta_omdb = requests.get(
    URL_OMDB,
    params={
        'apikey': OMDB_API_KEY,
        't': 'A Minecraft Movie',
        'y': 2025,
        'type': 'movie',
        'plot': 'short',
    },
    timeout=30,
)
respuesta_omdb.raise_for_status()
detalles_omdb = respuesta_omdb.json()

if detalles_omdb.get('Response') == 'False':
    raise ValueError(f"OMDb no devolvió una película válida: {detalles_omdb.get('Error')}")

campos_omdb = ['Title', 'Year', 'Released', 'imdbID', 'imdbRating', 'imdbVotes', 'Metascore', 'BoxOffice', 'Ratings']
pd.DataFrame([{campo: detalles_omdb.get(campo) for campo in campos_omdb}])

,Title,Year,Released,imdbID,imdbRating,imdbVotes,Metascore,BoxOffice,Ratings
0,A Minecraft Movie,2025,04 Apr 2025,tt3566834,5.6,"168,568",45,"$424,087,780","[{'Source': 'Internet Movie Database', 'Value'..."


## Guardado de datos crudos

Las celdas siguientes guardan exactamente las respuestas recibidas, sin transformarlas. Los archivos no contienen las API keys: estas solo viajan en la consulta y permanecen en `.env`.

In [7]:
def guardar_json(datos: dict, ruta: Path) -> None:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with ruta.open('w', encoding='utf-8') as archivo:
        json.dump(datos, archivo, ensure_ascii=False, indent=2)

fecha_extraccion = datetime.now(timezone.utc).date().isoformat()
ruta_tmdb = RAIZ_PROYECTO / 'data' / 'raw' / 'tmdb' / f'minecraft_movie_{fecha_extraccion}.json'
ruta_omdb = RAIZ_PROYECTO / 'data' / 'raw' / 'omdb' / f'minecraft_movie_{fecha_extraccion}.json'
ruta_metadata = RAIZ_PROYECTO / 'data' / 'raw' / f'metadata_{fecha_extraccion}.json'

metadata_extraccion = {
    'consulted_at_utc': datetime.now(timezone.utc).isoformat(),
    'movie': {'tmdb_id': PELICULA_ID_TMDB, 'title': 'A Minecraft Movie', 'year': 2025},
    'sources': [
        {'name': 'TMDB', 'endpoint': f'/movie/{PELICULA_ID_TMDB}', 'parameters': {'language': 'es-AR'}},
        {'name': 'OMDb', 'endpoint': '/', 'parameters': {'t': 'A Minecraft Movie', 'y': 2025, 'type': 'movie', 'plot': 'short'}},
    ],
}

guardar_json(detalles_tmdb, ruta_tmdb)
guardar_json(detalles_omdb, ruta_omdb)
guardar_json(metadata_extraccion, ruta_metadata)

print(f'TMDB guardado en: {ruta_tmdb.relative_to(RAIZ_PROYECTO)}')
print(f'OMDb guardado en: {ruta_omdb.relative_to(RAIZ_PROYECTO)}')
print(f'Metadatos guardados en: {ruta_metadata.relative_to(RAIZ_PROYECTO)}')

TMDB guardado en: data\raw\tmdb\minecraft_movie_2026-11-10.json
OMDb guardado en: data\raw\omdb\minecraft_movie_2026-11-10.json
Metadatos guardados en: data\raw\metadata_2026-11-10.json
